# 残基ごとの RSCC 計算

PDB ID を入力として、RCSB から PDB ファイルと Structure Factor を取得し、
Phenix `real_space_correlation` で残基ごとの Real-Space Correlation Coefficient (RSCC) を計算します。

**出力 CSV 列:** chain_id, residue_number, residue_name, rscc

**必要なソフトウェア (Mac):**
- [Phenix](https://phenix-online.org/) (`phenix.real_space_correlation`)
- [CCP4](https://www.ccp4.ac.uk/) または [GEMMI](https://github.com/project-gemmi/gemmi) (`gemmi cif2mtz`)

各セルは独立したステップです。順番に実行し、中間結果を確認してください。

## Step 0: Python 依存パッケージ

初回のみ実行してください（`pandas`, `matplotlib`）。

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = ["pandas", "matplotlib"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required Python packages are already installed.")


## Step 1: 入力パラメータの設定

PDB ID のみを指定します（4文字、大文字小文字は問いません）。

In [ ]:
from pathlib import Path

# ===== ここを変更 =====
PDB_ID = "3klr"
# =====================

PDB_ID = PDB_ID.strip().lower()
if len(PDB_ID) != 4 or not PDB_ID.isalnum():
    raise ValueError(f"Invalid PDB ID: {PDB_ID!r} (expected 4 alphanumeric characters)")

WORK_DIR = Path("work") / PDB_ID
WORK_DIR.mkdir(parents=True, exist_ok=True)

PDB_PATH = WORK_DIR / f"{PDB_ID}.pdb"
SF_CIF_PATH = WORK_DIR / f"{PDB_ID}-sf.cif"
MTZ_PATH = WORK_DIR / f"{PDB_ID}.mtz"
CSV_PATH = WORK_DIR / f"{PDB_ID}_rscc.csv"

print(f"PDB ID     : {PDB_ID.upper()}")
print(f"Work dir   : {WORK_DIR.resolve()}")
print(f"PDB file   : {PDB_PATH.name}")
print(f"SF CIF     : {SF_CIF_PATH.name}")
print(f"MTZ file   : {MTZ_PATH.name}")
print(f"Output CSV : {CSV_PATH.name}")

## Step 2: 実行環境の確認

Phenix と GEMMI (`cif2mtz`) のパスを自動検出します。

In [ ]:
import glob
import shutil
from typing import Optional


def find_executable(name: str, search_patterns: list[str]) -> Optional[str]:
    """PATH または典型的な Mac のインストール先から実行ファイルを探す。"""
    found = shutil.which(name)
    if found:
        return found
    for pattern in search_patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return matches[-1]
    return None


PHENIX_RSCC = find_executable(
    "phenix.real_space_correlation",
    ["/Applications/phenix-*/build/bin/phenix.real_space_correlation"],
)
GEMMI = find_executable(
    "gemmi",
    ["/Applications/ccp4-*/bin/gemmi", "/usr/local/bin/gemmi"],
)

print("=== Environment ===")
print(f"phenix.real_space_correlation : {PHENIX_RSCC or 'NOT FOUND'}")
print(f"gemmi                         : {GEMMI or 'NOT FOUND'}")

missing = []
if not PHENIX_RSCC:
    missing.append("Phenix (phenix.real_space_correlation)")
if not GEMMI:
    missing.append("GEMMI (gemmi cif2mtz)")

if missing:
    raise RuntimeError(
        "Required tools not found:\n  - " + "\n  - ".join(missing) + "\n"
        "Install Phenix and CCP4/GEMMI, or add them to PATH."
    )

print("\nAll required tools are available.")

## Step 3: RCSB から PDB ファイルをダウンロード

URL: `https://files.rcsb.org/download/{PDB_ID}.pdb`

In [ ]:
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

RCSB_BASE_URL = "https://files.rcsb.org/download"


def download_file(url: str, output_path: Path, overwrite: bool = False) -> Path:
    if output_path.exists() and not overwrite:
        print(f"Skip (already exists): {output_path}")
        return output_path

    try:
        with urlopen(url, timeout=60) as response:
            data = response.read()
    except HTTPError as exc:
        raise FileNotFoundError(f"Download failed ({exc.code}): {url}") from exc
    except URLError as exc:
        raise ConnectionError(f"Network error: {url} ({exc})") from exc

    output_path.write_bytes(data)
    print(f"Downloaded: {output_path} ({len(data):,} bytes)")
    return output_path


pdb_url = f"{RCSB_BASE_URL}/{PDB_ID}.pdb"
download_file(pdb_url, PDB_PATH)

print("\n--- PDB file preview (first 10 lines) ---")
for line in PDB_PATH.read_text(errors="replace").splitlines()[:10]:
    print(line)

## Step 4: RCSB から Structure Factor (mmCIF) をダウンロード

RCSB は MTZ を直接提供しないため、Structure Factor mmCIF (`-sf.cif`) を取得します。

URL: `https://files.rcsb.org/download/{PDB_ID}-sf.cif`

In [ ]:
sf_url = f"{RCSB_BASE_URL}/{PDB_ID}-sf.cif"
download_file(sf_url, SF_CIF_PATH)

print("\n--- Structure factor CIF preview (first 15 lines) ---")
for line in SF_CIF_PATH.read_text(errors="replace").splitlines()[:15]:
    print(line)

## Step 5: mmCIF → MTZ 変換

Phenix は MTZ 形式の反射データを要求するため、`gemmi cif2mtz` で変換します。

In [ ]:
import subprocess

if MTZ_PATH.exists():
    print(f"Skip (already exists): {MTZ_PATH}")
else:
    cmd = [GEMMI, "cif2mtz", str(SF_CIF_PATH), str(MTZ_PATH)]
    print("Command:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"gemmi cif2mtz failed (exit {result.returncode})")
    if result.stdout.strip():
        print(result.stdout)
    print(f"Created: {MTZ_PATH} ({MTZ_PATH.stat().st_size:,} bytes)")

mtz_info = subprocess.run(
    [GEMMI, "mtz", str(MTZ_PATH)],
    capture_output=True,
    text=True,
    check=True,
)
print("\n--- MTZ file info ---")
print(mtz_info.stdout[:2000] if len(mtz_info.stdout) > 2000 else mtz_info.stdout)

## Step 6: Phenix で残基ごとの RSCC を計算

`phenix.real_space_correlation` を `detail=residue` で実行します。

※ 構造のサイズによっては数分かかることがあります。

In [ ]:
cmd = [
    PHENIX_RSCC,
    str(PDB_PATH),
    str(MTZ_PATH),
    "detail=residue",
]
print("Command:", " ".join(cmd))
print("Running... (this may take a few minutes)\n")

result = subprocess.run(cmd, capture_output=True, text=True)
PHENIX_STDOUT = result.stdout
PHENIX_STDERR = result.stderr

if result.returncode != 0:
    print("=== STDERR ===")
    print(PHENIX_STDERR[-3000:] if len(PHENIX_STDERR) > 3000 else PHENIX_STDERR)
    print("\n=== STDOUT (tail) ===")
    print(PHENIX_STDOUT[-3000:] if len(PHENIX_STDOUT) > 3000 else PHENIX_STDOUT)
    raise RuntimeError(f"phenix.real_space_correlation failed (exit {result.returncode})")

if PHENIX_STDERR.strip():
    print("=== STDERR (informational) ===")
    print(PHENIX_STDERR[-1000:] if len(PHENIX_STDERR) > 1000 else PHENIX_STDERR)

print("\n--- Phenix output (last 25 lines) ---")
stdout_lines = PHENIX_STDOUT.splitlines()
for line in stdout_lines[-25:]:
    print(line)

print(f"\nTotal output lines: {len(stdout_lines)}")

## Step 7: Phenix 出力をパースして DataFrame を作成

残基テーブル (`<id string>  occ  ADP  CC  Rho1  Rho2`) を解析します。

In [ ]:
from dataclasses import dataclass
from typing import List

import pandas as pd

TABLE_HEADER = "<id string>"


@dataclass
class RsccRecord:
    chain_id: str
    altloc: str
    residue_name: str
    residue_number: str
    occ: float
    adp: float
    rscc: float
    rho1: float
    rho2: float


def parse_phenix_rscc_output(stdout: str) -> List[RsccRecord]:
    """phenix.real_space_correlation detail=residue の stdout から残基 RSCC を抽出。"""
    lines = stdout.splitlines()
    table_started = False
    records: List[RsccRecord] = []

    for line in lines:
        if TABLE_HEADER in line:
            table_started = True
            continue
        if not table_started or not line.strip():
            continue

        parts = line.split()
        if len(parts) == 8:
            chain_id, residue_name, residue_number, occ, adp, rscc, rho1, rho2 = parts
            altloc = ""
        elif len(parts) == 9:
            chain_id, altloc, residue_name, residue_number, occ, adp, rscc, rho1, rho2 = parts
        else:
            continue

        try:
            records.append(
                RsccRecord(
                    chain_id=chain_id,
                    altloc=altloc,
                    residue_name=residue_name,
                    residue_number=residue_number,
                    occ=float(occ),
                    adp=float(adp),
                    rscc=float(rscc),
                    rho1=float(rho1),
                    rho2=float(rho2),
                )
            )
        except ValueError:
            continue

    if not records:
        raise ValueError(
            "No residue RSCC records found in Phenix output. "
            "Check that detail=residue was used and the run completed successfully."
        )
    return records


records = parse_phenix_rscc_output(PHENIX_STDOUT)

df = pd.DataFrame(
    {
        "chain_id": [r.chain_id for r in records],
        "residue_number": [r.residue_number for r in records],
        "residue_name": [r.residue_name for r in records],
        "rscc": [r.rscc for r in records],
    }
)

print(f"Parsed {len(df)} residue records")
print(f"\nRSCC summary:\n{df['rscc'].describe()}")
print("\n--- First 10 rows ---")
df.head(10)

## Step 8: CSV 出力

列: `chain_id`, `residue_number`, `residue_name`, `rscc`

In [ ]:
OUTPUT_COLUMNS = ["chain_id", "residue_number", "residue_name", "rscc"]

df[OUTPUT_COLUMNS].to_csv(CSV_PATH, index=False)

print(f"Saved: {CSV_PATH.resolve()}")
print(f"Rows : {len(df)}")
print("\n--- CSV preview (first 10 lines) ---")
for line in CSV_PATH.read_text().splitlines()[:11]:
    print(line)

## (Optional) Step 9: RSCC の分布を可視化

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["rscc"], bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("RSCC")
ax.set_ylabel("Count")
ax.set_title(f"RSCC distribution ({PDB_ID.upper()})")
ax.axvline(df["rscc"].median(), color="red", linestyle="--", label=f"median={df['rscc'].median():.3f}")
ax.legend()
plt.tight_layout()
plt.show()